# Module 3: Memory & Persistence

**Day 4 — LangGraph Agents, Memory, HITL & MCP**

## What you will learn
- **InMemorySaver**: ephemeral in-process checkpointing
- **thread_id**: isolates different conversations
- **Conversation history**: how messages accumulate via `add_messages`
- **Time travel**: `get_state_history()` to replay or debug past states
- **SqliteSaver**: persist memory across process restarts

## Why persistence matters
Without a checkpointer, every `.invoke()` starts fresh — the bot forgets everything.
With `InMemorySaver + thread_id`, the bot remembers the entire conversation.


In [ ]:
import sys
sys.path.insert(0, '../src')
print('Path configured.')

## 1. Stateless vs Stateful Graph

Without a checkpointer, each invocation is completely independent.

In [ ]:
from day4.memory_persistence import build_chatbot_graph, chat_turn, get_conversation_history, get_state_history
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver

# Simple echo LLM
call_count = [0]
def echo_llm(messages):
    call_count[0] += 1
    last = messages[-1].content
    all_human = [m.content for m in messages if isinstance(m, HumanMessage)]
    return f'I remember {len(all_human)} message(s). You said: "{last[:40]}"'

# Stateless: each call is independent
stateless = build_chatbot_graph(echo_llm)
r1 = stateless.invoke({'messages': [HumanMessage('My name is Naval.')]})
r2 = stateless.invoke({'messages': [HumanMessage('What is my name?')]})

print('Stateless graph:')
print(f'  Turn 1: {r1["messages"][-1].content}')
print(f'  Turn 2: {r2["messages"][-1].content}')
print('  (No memory — each call starts fresh)')

## 2. InMemorySaver + thread_id

Adding a checkpointer gives the bot persistent memory within the same process run.

In [ ]:
# With InMemorySaver
checkpointer = MemorySaver()
call_count[0] = 0
chatbot = build_chatbot_graph(echo_llm, checkpointer=checkpointer)

thread_id = 'naval_session_001'

reply1 = chat_turn(chatbot, 'My name is Naval.', thread_id)
reply2 = chat_turn(chatbot, 'I work at a startup.', thread_id)
reply3 = chat_turn(chatbot, 'What have I told you so far?', thread_id)

print('With InMemorySaver (thread_id: naval_session_001):')
print(f'  Turn 1: {reply1}')
print(f'  Turn 2: {reply2}')
print(f'  Turn 3: {reply3}')

## 3. Conversation History

`get_conversation_history()` reads stored messages without running any nodes.

In [ ]:
history = get_conversation_history(chatbot, thread_id)
print(f'Total messages in thread: {len(history)}')
print()
for msg in history:
    role = msg['role'].upper()
    content = msg['content'][:60]
    print(f'  [{role:5s}] {content}')

## 4. Thread Isolation

Different thread_ids are completely isolated — one user's conversation never leaks to another.

In [ ]:
# Thread isolation
chat_turn(chatbot, 'Hi, I am Priya from Chennai.', 'priya_session')
chat_turn(chatbot, 'Hi, I am Rahul from Delhi.', 'rahul_session')

priya_history = get_conversation_history(chatbot, 'priya_session')
rahul_history = get_conversation_history(chatbot, 'rahul_session')

print('Thread isolation:')
print(f'  priya_session messages: {len(priya_history)}')
print(f'  rahul_session messages: {len(rahul_history)}')
print(f'  Priya said: {priya_history[0]["content"]}')
print(f'  Rahul said: {rahul_history[0]["content"]}')

## 5. Time Travel — State Snapshots

Every `.invoke()` creates a checkpoint. `get_state_history()` lets you replay past states.

In [ ]:
snapshots = get_state_history(chatbot, thread_id)
print(f'Total checkpoints for thread {thread_id}: {len(snapshots)}')
print()
for snap in snapshots:
    print(f'  Step {snap["step"]}: {snap["message_count"]} messages')

## 6. SqliteSaver (persistent across restarts)

InMemorySaver is lost when the process exits. SqliteSaver persists to disk.

In [ ]:
# SqliteSaver example (requires: pip install langgraph-checkpoint-sqlite aiosqlite)
# Uncomment to use:
#
# from langgraph.checkpoint.sqlite import SqliteSaver
# sqlite_checkpointer = SqliteSaver.from_conn_string('chatbot_memory.db')
# persistent_bot = build_chatbot_graph(echo_llm, checkpointer=sqlite_checkpointer)
#
# # This conversation persists across process restarts!
# chat_turn(persistent_bot, 'My name is Naval.', 'persistent_001')

print('SqliteSaver: memory persists across restarts.')
print('PostgresSaver: production-grade, multi-user memory.')
print('Databricks Delta: distributed checkpoints for large-scale agents.')

## Databricks Bridge

In [ ]:
# In Databricks, store conversation history in Delta Lake:
#
# from delta.tables import DeltaTable
# import json
#
# def save_conversation_to_delta(thread_id, history):
#     spark.createDataFrame([{
#         'thread_id': thread_id,
#         'history':   json.dumps(history),
#         'timestamp': datetime.now()
#     }]).write.format('delta').mode('append').save('/mnt/conversations/')
#
# # After each session:
# history = get_conversation_history(chatbot, thread_id)
# save_conversation_to_delta(thread_id, history)

print('Databricks: store conversation history in Delta Lake for analytics and audit.')